# 07 — Modelos balanceados / seleção do limiar
Regressão Logística, Random Forest e XGBoost; ajuste com F2-score apenas em 2024.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import joblib
ROOT=Path.cwd(); OUT=ROOT/"outputs"; DATA=ROOT/"data"/"processed"
SILVER=DATA/"prf_acidentes_silver_2020_2025.parquet"
for p in ["modelos","interpretabilidade","modelo_final"]:
    (OUT/p).mkdir(parents=True,exist_ok=True)
df=pd.read_parquet(SILVER)
df["br_categoria"]=pd.to_numeric(df.br,errors="coerce").astype("Int64").astype("string").fillna("Não informado")
flags=sorted(c for c in df if c.startswith("tracado_") and c!="tracado_via")
categoricas=["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"]
numericas=["km","mes","hora","fim_semana","qtd_caracteristicas_tracado","veiculos"]+flags
features=categoricas+numericas
proibidas={"pessoas","mortos","feridos_graves","feridos_leves","feridos","ilesos","ignorados","classificacao_acidente","acidente_fatal"}
assert set(features).isdisjoint(proibidas)
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
def pre(cat=categoricas,num=numericas):
    try: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse_output=True)
    except TypeError: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse=True)
    cat_pipe=Pipeline([("imputar",SimpleImputer(strategy="most_frequent")),("onehot",enc)])
    num_pipe=Pipeline([("imputar",SimpleImputer(strategy="median")),("escala",StandardScaler())])
    return ColumnTransformer([("categoricas",cat_pipe,cat),("numericas",num_pipe,num)])
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,fbeta_score,roc_auc_score,average_precision_score,confusion_matrix
def avaliar(y,proba,nome,limiar=0.5):
    pred=(proba>=limiar).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(modelo=nome,threshold=limiar,accuracy=accuracy_score(y,pred),
      precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),
      f1=f1_score(y,pred,zero_division=0),f2=fbeta_score(y,pred,beta=2,zero_division=0),
      roc_auc=roc_auc_score(y,proba),pr_auc=average_precision_score(y,proba),
      verdadeiro_negativo=tn,falso_positivo=fp,falso_negativo=fn,verdadeiro_positivo=tp)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
treino=df[df.ano<=2023];valid=df[df.ano==2024]
X=treino[features];y=treino.acidente_grave
Xv=valid[features];yv=valid.acidente_grave
peso=(y==0).sum()/(y==1).sum()
estimadores={
    "Logística Balanceada":LogisticRegression(solver="saga",max_iter=700,class_weight="balanced",random_state=42),
    "Random Forest":RandomForestClassifier(n_estimators=300,max_depth=20,min_samples_leaf=10,class_weight="balanced_subsample",n_jobs=-1,random_state=42),
    "XGBoost":XGBClassifier(n_estimators=500,learning_rate=.05,max_depth=6,min_child_weight=5,subsample=.8,colsample_bytree=.8,scale_pos_weight=peso,objective="binary:logistic",eval_metric="logloss",n_jobs=-1,random_state=42)
}
modelos={};probabilidades={};linhas=[];curvas=[]
for nome,est in estimadores.items():
    clf=Pipeline([("preprocessamento",pre()),("modelo",est)])
    clf.fit(X,y)
    modelos[nome]=clf
    joblib.dump(clf,OUT/"modelos"/{"XGBoost":"xgboost.joblib","Random Forest":"random_forest.joblib","Logística Balanceada":"logistica_balanceada.joblib"}[nome])
    prob=clf.predict_proba(Xv)[:,1];probabilidades[nome]=prob
    linhas.append(avaliar(yv,prob,nome,.5))
    for t in np.arange(.10,.8001,.02):
        r=avaliar(yv,prob,nome,float(round(t,2)))
        curvas.append({k:r[k] for k in ["modelo","threshold","precision","recall","f1","f2"]})
comp=pd.DataFrame(linhas).sort_values("f2",ascending=False)
curva=pd.DataFrame(curvas)
melhores=curva.loc[curva.groupby("modelo").f2.idxmax()].reset_index(drop=True)
display(comp);display(melhores)
comp.to_csv(OUT/"modelos"/"comparacao_modelos_balanceados.csv",index=False)
curva.to_csv(OUT/"modelos"/"analise_thresholds.csv",index=False)
melhores.to_csv(OUT/"modelos"/"melhores_thresholds.csv",index=False)
print("Limiar histórico do estudo: 0.28; confira eventuais diferenças após reexecução.")
print("Decisão congelada: XGBoost, limiar 0.28; teste 2025 segue reservado.")